# DIAGNÓSTICO WEB SEARCH — VERSIÓN CANÓNICA

**Versión:** `v0.9-websearch-smoke-test`  
**Última modificación:** `2026-09-11 20:14 CEST`  
**Rama:** `fix/chapter2-news-agent`  
**Modelo:** `gpt-4.1-mini`

Esta versión **no ejecuta todavía el Evaluator-Optimizer**. Primero comprueba de forma aislada si el hosted `web_search` de Responses API recupera fuentes generales y si puede recuperar Reuters. Así evitamos gastar llamadas en un bucle que no puede funcionar si la recuperación devuelve cero fuentes.


## 1. Instalar dependencias

In [13]:
!pip install -U openai openai-agents -q

## 2. Entorno y helpers

In [14]:
import json, os, sys, importlib.metadata as im
from urllib.parse import urlsplit
from google.colab import userdata
from openai import OpenAI, BadRequestError

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()

print("Python:", sys.version)
print("openai:", im.version("openai"))
print("openai-agents:", im.version("openai-agents"))
print("API key presente:", bool(os.environ.get("OPENAI_API_KEY")))

def to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return obj

def collect_urls(obj):
    obj = to_dict(obj)
    found=[]
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k in {"url","source_url","source_website_url"} and isinstance(v,str) and v.startswith("http"):
                found.append(v)
            found.extend(collect_urls(v))
    elif isinstance(obj,(list,tuple)):
        for v in obj:
            found.extend(collect_urls(v))
    return list(dict.fromkeys(found))

def print_web_calls(resp):
    calls=[x for x in resp.output if getattr(x,"type",None)=="web_search_call"]
    print("web_search_call count:",len(calls))
    for i,c in enumerate(calls,1):
        a=getattr(c,"action",None)
        print(f"  CALL {i} status:",getattr(c,"status",None))
        print("    query:",getattr(a,"query",None))
        print("    queries:",getattr(a,"queries",None))
        print("    sources:",getattr(a,"sources",None))
        print("    results:",getattr(c,"results",None))

def run_probe(label, query, allowed_domains=None):
    print("\n"+"#"*90)
    print(label)
    print("QUERY:",query)
    tool={"type":"web_search","search_context_size":"high","external_web_access":True}
    if allowed_domains:
        tool["filters"]={"allowed_domains":allowed_domains}
    includes=["web_search_call.action.sources","web_search_call.results"]
    try:
        resp=client.responses.create(model="gpt-4.1-mini",tools=[tool],tool_choice="required",include=includes,input=query)
    except BadRequestError as e:
        print("El API no aceptó web_search_call.results; reintentando solo con action.sources")
        print("BadRequest:",str(e)[:1000])
        resp=client.responses.create(model="gpt-4.1-mini",tools=[tool],tool_choice="required",include=["web_search_call.action.sources"],input=query)
    print("OUTPUT TEXT:",resp.output_text)
    print("OUTPUT TYPES:",[getattr(x,"type",None) for x in resp.output])
    print_web_calls(resp)
    urls=collect_urls(resp)
    reuters=[u for u in urls if "reuters.com" in (urlsplit(u).hostname or "").lower()]
    print("TOTAL URLS RECUPERADAS:",len(urls))
    for u in urls[:30]: print("  URL:",u)
    print("REUTERS URLS:",len(reuters))
    for u in reuters: print("  REUTERS:",u)
    print("tool_usage:",getattr(resp,"tool_usage",None))
    return {"label":label,"response":resp,"urls":urls,"reuters":reuters}


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
openai: 3.13.0
openai-agents: 0.22.2
API key presente: True


## 3. Matriz de pruebas

Ejecuta esta celda completa. Las pruebas van de lo más general a Reuters específico. Dos de los titulares Reuters son artículos reales conocidos del 10 de septiembre de 2026 y se usan **solo como prueba de indexación/recuperación**.

In [15]:
probes=[]

probes.append(run_probe(
    "A1 — WEB GENERAL / OPENAI DOCS",
    "Find the official OpenAI documentation page about web search in the Responses API. Cite the source."
))

probes.append(run_probe(
    "A2 — WEB GENERAL / NOTICIAS IA ACTUALES",
    "Find three current artificial intelligence news articles published today. Cite the sources."
))

probes.append(run_probe(
    "A3 — REUTERS SIN RESTRICCIÓN DE FECHA",
    "Find one Reuters article about artificial intelligence or OpenAI. Cite the direct Reuters source."
))

probes.append(run_probe(
    "A4 — REUTERS / TITULAR REAL OPENAI",
    "Find the Reuters article titled 'OpenAI launches ChatGPT for financial services industry' published September 10, 2026. Cite the direct Reuters source."
))

probes.append(run_probe(
    "A5 — REUTERS / TITULAR REAL ADOBE",
    "Find the Reuters article titled 'Adobe beats third-quarter revenue estimates on AI product demand' published September 10, 2026. Cite the direct Reuters source."
))

probes.append(run_probe(
    "A6 — REUTERS CON FILTRO DE DOMINIO",
    "Find the Reuters article titled 'OpenAI launches ChatGPT for financial services industry' published September 10, 2026.",
    allowed_domains=["reuters.com"]
))



##########################################################################################
A1 — WEB GENERAL / OPENAI DOCS
QUERY: Find the official OpenAI documentation page about web search in the Responses API. Cite the source.
OUTPUT TEXT: You can find the official OpenAI documentation on web search in the Responses API at the following link: ([developer-openai-com.sitemirror.store](https://developer-openai-com.sitemirror.store/api/docs/guides/tools-web-search/?utm_source=openai)) 
OUTPUT TYPES: ['web_search_call', 'message']
web_search_call count: 1
  CALL 1 status: completed
    query: Find the official OpenAI documentation page about web search in the Responses API. Cite the source.
    queries: ['Find the official OpenAI documentation page about web search in the Responses API. Cite the source.']
    sources: [ActionSearchSource(type='url', url='https://openai.com/index/new-tools-for-building-agents/?utm_source=openai'), ActionSearchSource(type='url', url='https://developers.ope

BadRequestError: Error code: 400 - {'error': {'message': "Parameter 'filters' not supported with model 'gpt-4.1-mini'", 'type': 'invalid_request_error', 'param': 'tools', 'code': None}}

## 4. Diagnóstico automático

In [ ]:
general_urls=len(probes[0]["urls"])+len(probes[1]["urls"])
reuters_urls=sum(len(p["reuters"]) for p in probes[2:])

print("\n"+"="*90)
print("DIAGNÓSTICO FINAL")
print("URLs generales recuperadas (A1+A2):",general_urls)
print("URLs Reuters recuperadas (A3-A6):",reuters_urls)

if general_urls==0:
    print("RESULTADO: FALLO GENERAL DE HOSTED WEB SEARCH en este proyecto/API key/modelo.")
    print("El problema no está en Agents SDK ni en el Evaluator-Optimizer. Responses API ejecuta la herramienta pero no devuelve fuentes.")
elif reuters_urls==0:
    print("RESULTADO: web_search funciona en general, pero NO está recuperando Reuters.")
    print("El siguiente paso es sustituir la capa de retrieval de Reuters por otra fuente/buscador y mantener gpt-4.1-mini para síntesis/evaluación.")
else:
    print("RESULTADO: retrieval funciona y Reuters es accesible. Ya podemos volver al Evaluator-Optimizer y depurar solo la síntesis/evaluación.")
